In [22]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    classification_report, 
    confusion_matrix, 
    make_scorer, 
    recall_score, 
    precision_score,
    f1_score
    )
from sklearn.model_selection import GridSearchCV


### 데이터 불러오기

In [23]:
# 파일 불러오기 
target = "Risk_Label"

# Load datasets
train_df = pd.read_csv(r"../../data/processed/ADASYN/data_train_adasyn.csv")
valid_df = pd.read_csv(r"../../data/processed/ADASYN/data_valid.csv")
test_df = pd.read_csv(r"../../data/processed/ADASYN/data_test.csv")

# Split features/target
X_train = train_df.drop(columns=[target])
y_train = train_df[target]

X_valid = valid_df.drop(columns=[target])
y_valid = valid_df[target]

X_test = test_df.drop(columns=[target])
y_test = test_df[target]


print("train shape:", X_train.shape, y_train.shape)
print("valid shape:", X_valid.shape, y_valid.shape)
print("test shape:", X_test.shape, y_test.shape)

train shape: (2466, 9) (2466,)
valid shape: (1438, 9) (1438,)
test shape: (822, 9) (822,)


## Grid Search SVM 적합

In [24]:
# SVM pipeline (no scaler)
svm_pipeline = Pipeline(                                                                                                       
    steps=[("svm", SVC())]
)

# Grid search space
param_grid = {
    "svm__kernel": ["rbf", "linear", "poly"],
    "svm__C": [0.01, 0.1, 1, 10, 50],
    "svm__gamma": ["scale", "auto", 0.01, 0.1],
    "svm__class_weight": [None],
}


def g_mean_scorer(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    
    return np.sqrt(sensitivity * specificity)

# Grid search - scoring은 recall 사용 (CV 평가용)
# 최종 best 모델은 validation g_mean으로 선택
highrisk_recall = make_scorer(recall_score, pos_label=1)

grid_search = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=param_grid,
    scoring=highrisk_recall,
    cv=5,
    n_jobs=-1,
    verbose=1,
    refit=False  # refit 안함 - 수동으로 best 모델 선택
)

grid_search.fit(X_train, y_train)


Fitting 5 folds for each of 60 candidates, totalling 300 fits


GridSearchCV(cv=5, estimator=Pipeline(steps=[('svm', SVC())]), n_jobs=-1,
             param_grid={'svm__C': [0.01, 0.1, 1, 10, 50],
                         'svm__class_weight': [None],
                         'svm__gamma': ['scale', 'auto', 0.01, 0.1],
                         'svm__kernel': ['rbf', 'linear', 'poly']},
             refit=False,
             scoring=make_scorer(recall_score, response_method='predict', pos_label=1),
             verbose=1)

### 지표

In [25]:
# 각 파라미터 조합에 대해 validation H1 스코어 계산
best_valid_h1 = -1.0
best_idx = 0

for i, params in enumerate(grid_search.cv_results_["params"]):
    model = grid_search.estimator.set_params(**params)
    model.fit(X_train, y_train)
    val_pred = model.predict(X_valid)
    
    cm = confusion_matrix(y_valid, val_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    
    # F1, G-Mean, H1 계산
    val_f1 = f1_score(y_valid, val_pred, zero_division=0)
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    valid_gm = np.sqrt(sensitivity * specificity)
    
    # H1 스코어: F1과 G-Mean의 조화 평균
    valid_h1 = float(2 * val_f1 * valid_gm / (val_f1 + valid_gm)) if (val_f1 + valid_gm) > 0 else 0.0
    
    if valid_h1 > best_valid_h1:
        best_valid_h1 = valid_h1
        best_idx = i

# best 모델 선택
best_params = grid_search.cv_results_["params"][best_idx]
best_svm = svm_pipeline.set_params(**best_params)
best_svm.fit(X_train, y_train)

valid_pred = best_svm.predict(X_valid)

# Validation 지표 계산
cm_valid = confusion_matrix(y_valid, valid_pred, labels=[0, 1])
tn, fp, fn, tp = cm_valid.ravel()

valid_acc = accuracy_score(y_valid, valid_pred)
valid_f1 = f1_score(y_valid, valid_pred, zero_division=0)
valid_recall = recall_score(y_valid, valid_pred, zero_division=0)
valid_precision = precision_score(y_valid, valid_pred, zero_division=0)
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
valid_g_mean = np.sqrt(sensitivity * specificity)

# H1 스코어 계산
valid_h1 = float(2 * valid_f1 * valid_g_mean / (valid_f1 + valid_g_mean)) if (valid_f1 + valid_g_mean) > 0 else 0.0

print("\n" + "="*60)
print("Best Model (by validation H1-Score)")
print("="*60)
print("Best Params:", best_params)
print("Best Valid H1-Score:", best_valid_h1)

print("\n[Validation] Metrics")
print(f"Accuracy  : {valid_acc:.4f}")
print(f"F1-Score  : {valid_f1:.4f}")
print(f"Recall    : {valid_recall:.4f}")
print(f"Precision : {valid_precision:.4f}")
print(f"G-Mean    : {valid_g_mean:.4f}")
print(f"H1-Score  : {valid_h1:.4f}")
print(f"\n[Validation] Confusion Matrix:\n", cm_valid)


Best Model (by validation H1-Score)
Best Params: {'svm__C': 0.1, 'svm__class_weight': None, 'svm__gamma': 'scale', 'svm__kernel': 'poly'}
Best Valid H1-Score: 0.4533458417909031

[Validation] Metrics
Accuracy  : 0.8679
F1-Score  : 0.3750
Recall    : 0.3519
Precision : 0.4014
G-Mean    : 0.5731
H1-Score  : 0.4533

[Validation] Confusion Matrix:
 [[1191   85]
 [ 105   57]]


### Test 성능 결과 

In [26]:
# Test prediction/evaluation using tuned model
test_pred = best_svm.predict(X_test)

# Test 지표 계산
cm_test = confusion_matrix(y_test, test_pred, labels=[0, 1])
tn, fp, fn, tp = cm_test.ravel()

test_acc = accuracy_score(y_test, test_pred)
test_f1 = f1_score(y_test, test_pred, zero_division=0)
test_recall = recall_score(y_test, test_pred, zero_division=0)
test_precision = precision_score(y_test, test_pred, zero_division=0)
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
test_g_mean = np.sqrt(sensitivity * specificity)

# H1 스코어 계산
test_h1 = float(2 * test_f1 * test_g_mean / (test_f1 + test_g_mean)) if (test_f1 + test_g_mean) > 0 else 0.0

print("\n" + "="*60)
print("[Test] Metrics")
print("="*60)
print(f"Accuracy  : {test_acc:.4f}")
print(f"F1-Score  : {test_f1:.4f}")
print(f"Recall    : {test_recall:.4f}")
print(f"Precision : {test_precision:.4f}")
print(f"G-Mean    : {test_g_mean:.4f}")
print(f"H1-Score  : {test_h1:.4f}")
print(f"\n[Test] Confusion Matrix:\n", cm_test)
print("\nClassification Report:")
from sklearn.metrics import classification_report
print(classification_report(y_test, test_pred, digits=4, zero_division=0))

# Save predictions
valid_pred_df = valid_df.copy()
valid_pred_df["SVM_Prob"] = 0  # SVM은 probability 제공 안하므로 0
valid_pred_df["SVM_Pred"] = valid_pred

test_pred_df = test_df.copy()
test_pred_df["SVM_Prob"] = 0  # SVM은 probability 제공 안하므로 0
test_pred_df["SVM_Pred"] = test_pred

valid_pred_df.to_csv("../../data/processed/result/SVM_valid_pred.csv", index=False)
test_pred_df.to_csv("../../data/processed/result/SVM_test_pred.csv", index=False)

print("\n✓ \"SVM_valid_pred.csv\", \"SVM_test_pred.csv\" 파일로 저장 완료")


[Test] Metrics
Accuracy  : 0.8723
F1-Score  : 0.3478
Recall    : 0.3077
Precision : 0.4000
G-Mean    : 0.5385
H1-Score  : 0.4227

[Test] Confusion Matrix:
 [[689  42]
 [ 63  28]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9162    0.9425    0.9292       731
           1     0.4000    0.3077    0.3478        91

    accuracy                         0.8723       822
   macro avg     0.6581    0.6251    0.6385       822
weighted avg     0.8591    0.8723    0.8648       822


✓ "SVM_valid_pred.csv", "SVM_test_pred.csv" 파일로 저장 완료


In [28]:
print(best_valid_h1, test_h1)
print(confusion_matrix(y_valid, valid_pred, labels=[0, 1]))
print(confusion_matrix(y_test, test_pred, labels=[0, 1]))

0.4533458417909031 0.422662344661505
[[1191   85]
 [ 105   57]]
[[689  42]
 [ 63  28]]
